# Aerial Object Classification & Detection
## Notebook 1 — EDA & Data Preprocessing

In this notebook I'm going to explore the dataset, understand the structure,
check for any imbalance, and then prepare the data for model training.

**Dataset:** Bird vs Drone aerial images

**Task:** Binary Image Classification

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# base path where my dataset folders are stored
base = '/content/drive/MyDrive/'

print('Drive mounted successfully!')

In [ ]:
# checking what folders are available
folders = [
    'train-20260409T091319Z-3-001',
    'valid-20260409T091319Z-3-001',
    'test-20260409T091319Z-3-001']

for folder in folders:
    path = os.path.join(base, folder)
    print(f'\n {folder}:')
    for subfolder in os.listdir(path):
        sub_path = os.path.join(path, subfolder)
        if os.path.isdir(sub_path):
            print(f'   -- {subfolder}:')
            for item in os.listdir(sub_path):
                print(f'      -- {item}')

In [ ]:
TRAIN_DIR = os.path.join(base, 'train-20260409T091319Z-3-001/train/')
VALID_DIR = os.path.join(base, 'valid-20260409T091319Z-3-001/valid/')
TEST_DIR  = os.path.join(base, 'test-20260409T091319Z-3-001/test/')

print('Paths set successfully!')
print('TRAIN:', TRAIN_DIR)
print('VALID:', VALID_DIR)
print('TEST :', TEST_DIR)

In [ ]:
# counting images in each class
for split_name, split_path in [('TRAIN', TRAIN_DIR), ('VALID', VALID_DIR), ('TEST', TEST_DIR)]:
    print(f'\n {split_name}:')
    for cls in ['bird', 'drone']:
        cls_path = os.path.join(split_path, cls)
        count = len(os.listdir(cls_path))
        print(f'   {cls}: {count} images')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

splits = ['TRAIN', 'VALID', 'TEST']
paths  = [TRAIN_DIR, VALID_DIR, TEST_DIR]

bird_counts  = []
drone_counts = []

for path in paths:
    bird_counts.append(len(os.listdir(os.path.join(path, 'bird'))))
    drone_counts.append(len(os.listdir(os.path.join(path, 'drone'))))

x = np.arange(len(splits))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - width/2, bird_counts,  width, label='Bird',  color='steelblue')
bars2 = ax.bar(x + width/2, drone_counts, width, label='Drone', color='tomato')

ax.set_title('Class Distribution Across Splits')
ax.set_xticks(x)
ax.set_xticklabels(splits)
ax.set_ylabel('Number of Images')
ax.legend()

# adding count labels on top of each bar
for bar in bars1 + bars2:
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 10,
            str(int(bar.get_height())),
            ha='center', fontsize=9)

plt.tight_layout()
plt.show()

# slight imbalance noticed — bird has more images than drone in training

In [ ]:
from PIL import Image
import random

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Sample Images — Bird (top) vs Drone (bottom)', fontsize=13)

for idx, cls in enumerate(['bird', 'drone']):
    cls_path = os.path.join(TRAIN_DIR, cls)
    # picking 5 random images from each class
    images = random.sample(os.listdir(cls_path), 5)
    for j, img_name in enumerate(images):
        img = Image.open(os.path.join(cls_path, img_name)).resize((224, 224))
        axes[idx][j].imshow(img)
        axes[idx][j].set_title(cls.upper())
        axes[idx][j].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
print('Checking sample image sizes...\n')

for cls in ['bird', 'drone']:
    cls_path = os.path.join(TRAIN_DIR, cls)
    sample_images = random.sample(os.listdir(cls_path), 5)
    print(f'Class: {cls.upper()}')
    for img_name in sample_images:
        img = Image.open(os.path.join(cls_path, img_name))
        print(f'   {img_name} -- Size: {img.size}, Mode: {img.mode}')
    print()

# images have different sizes so resizing to 224x224 is important

 Dataset Summary

In [ ]:
total_train = len(os.listdir(os.path.join(TRAIN_DIR, 'bird'))) + \
              len(os.listdir(os.path.join(TRAIN_DIR, 'drone')))
total_valid = len(os.listdir(os.path.join(VALID_DIR, 'bird'))) + \
              len(os.listdir(os.path.join(VALID_DIR, 'drone')))
total_test  = len(os.listdir(os.path.join(TEST_DIR,  'bird'))) + \
              len(os.listdir(os.path.join(TEST_DIR,  'drone')))

print('=' * 35)
print('DATASET SUMMARY')
print('=' * 35)
print(f'Total Train Images : {total_train}')
print(f'Total Valid Images : {total_valid}')
print(f'Total Test  Images : {total_test}')
print(f'Total Images       : {total_train + total_valid + total_test}')
print(f'Classes            : Bird, Drone')
print('=' * 35)

 Data Preprocessing & Augmentation


In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE   = (224, 224)
BATCH_SIZE = 32

# applying augmentation to training data to prevent overfitting
train_datagen = ImageDataGenerator(
    rescale=1./255,           # normalize pixel values to [0,1]
    rotation_range=20,        # randomly rotate images
    horizontal_flip=True,     # flip horizontally
    zoom_range=0.2,           # random zoom
    brightness_range=[0.8, 1.2],  # vary brightness
    shear_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1
)

# only normalization for validation and test — no augmentation
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='binary', shuffle=True
)

val_gen = val_test_datagen.flow_from_directory(
    VALID_DIR, target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='binary', shuffle=False
)

test_gen = val_test_datagen.flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='binary', shuffle=False
)

print('Generators created!')
print(f'Class Indices: {train_gen.class_indices}')

In [ ]:
images, labels = next(train_gen)

print('=' * 35)
print('BATCH INFO')
print('=' * 35)
print(f'Image Batch Shape : {images.shape}')
print(f'Label Batch Shape : {labels.shape}')
print(f'Pixel Value Range : {images.min():.2f} to {images.max():.2f}')
print(f'Sample Labels     : {labels[:5]}')
print('=' * 35)
print('Preprocessing complete! Ready for model training.')